In [15]:
%%writefile exceptions.py
class UserManagementError(Exception):
    """Base class for exceptions in this module."""
    pass

class UserAlreadyExistsError(UserManagementError):
    pass

class InvalidCredentialsError(UserManagementError):
    pass

class AccountLockedError(UserManagementError):
    pass

Writing exceptions.py


In [16]:
%%writefile file_handler.py
import json
import os

DB_FILE = "users.json"

def read_users():
    if not os.path.exists(DB_FILE):
        return {}
    with open(DB_FILE, 'r') as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return {}

def save_users(users):
    with open(DB_FILE, 'w') as f:
        json.dump(users, f, indent=4)

Writing file_handler.py


In [17]:
%%writefile log.py
from datetime import datetime

LOG_FILE = "activity.log"

def log_activity(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_FILE, 'a') as f:
        f.write(f"[{timestamp}] {message}\n")

Writing log.py


In [18]:
%%writefile utils.py
import base64
import re

def encrypt_password(password):
    return base64.b64encode(password.encode()).decode()

def decrypt_password(encoded_password):
    return base64.b64decode(encoded_password.encode()).decode()

def validate_password(password):
    # Min 6 chars, 1 number, 1 letter
    if len(password) < 6:
        return False
    if not re.search("[a-z]", password) or not re.search("[0-9]", password):
        return False
    return True

Writing utils.py


In [19]:
%%writefile auth.py
from file_handler import read_users, save_users
from utils import encrypt_password, validate_password
from exceptions import UserAlreadyExistsError, InvalidCredentialsError
from log import log_activity

def register_user(username, password):
    users = read_users()
    if username in users:
        raise UserAlreadyExistsError(f"User {username} already exists!")

    if not validate_password(password):
        print("Password too weak! Password must be at least 6 characters long and contain at least one letter and one number.")
        return False

    users[username] = {"password": encrypt_password(password), "role": "user"}
    save_users(users)
    log_activity(f"New user registered: {username}")
    return True

def login(username, password):
    users = read_users()
    if username not in users:
        raise InvalidCredentialsError("Invalid username or password.")

    if users[username]['password'] == encrypt_password(password):
        log_activity(f"User logged in: {username}")
        return users[username]
    else:
        raise InvalidCredentialsError("Invalid username or password.")

Writing auth.py


In [20]:
%%writefile main.py
import auth
from exceptions import UserManagementError

def user_dashboard(username, user_info):
    while True:
        print(f"\n--- {username}'s Dashboard ---")
        print("1. View Profile\n2. Update Password\n3. Delete Account\n4. Logout")
        db_choice = input("Select: ")

        if db_choice == '1':
            print(f"\n[Profile]\nUsername: {username}\nRole: {user_info.get('role', 'user')}")

        elif db_choice == '2':
            new_p = input("Enter new password: ")
            # You would call a function in auth.py to update the file here
            print("Password updated successfully!")

        elif db_choice == '3':
            confirm = input("Are you sure? (yes/no): ")
            if confirm.lower() == 'yes':
                # Call a delete function in auth.py
                print("Account deleted.")
                break

        elif db_choice == '4':
            print("Logging out...")
            break

def main():
    while True:
        print("\n--- Smart User Management ---")
        print("1. Register\n2. Login\n3. Exit")

        # .strip().lower() handles extra spaces and capital letters
        choice = input("Select: ").strip().lower()

        try:
            # Check for number OR the word
            if choice in ['1', 'register']:
                u = input("Username: ")
                p = input("Password: ")
                if auth.register_user(u, p):
                    print("Registration successful!")

            elif choice in ['2', 'login']:
                attempts = 0
                while attempts < 3:
                    u = input("Username: ")
                    p = input("Password: ")
                    try:
                        user_data = auth.login(u, p)
                        print(f"Welcome back, {u}!")
                        user_dashboard(u, user_data) # Call dashboard upon successful login
                        break # Exit login attempt loop
                    except UserManagementError as e: # Catch only UserManagementError for login attempts
                        attempts += 1
                        print(f"Error: {e}. Attempts left: {3 - attempts}")
                else:
                    print("Account locked temporarily due to too many failed attempts.")

            elif choice in ['3', 'exit']:
                print("Goodbye!")
                break

            else:
                print("Invalid selection. Please choose 1, 2, or 3.")

        except UserManagementError as e: # Catch UserManagementError for top-level issues like registration
            print(f"App Error: {e}")
        finally:
            print("Action processed.")

if __name__ == "__main__":
    main()

Writing main.py


In [21]:
!python main.py


--- Smart User Management ---
1. Register
2. Login
3. Exit
Select: 1
Username: epsi
Password: 1234@epsi
Registration successful!
Action processed.

--- Smart User Management ---
1. Register
2. Login
3. Exit
Select: 2
Username: epsi
Password: 1234@epsi
Welcome back, epsi!

--- epsi's Dashboard ---
1. View Profile
2. Update Password
3. Delete Account
4. Logout
Select: 1

[Profile]
Username: epsi
Role: user

--- epsi's Dashboard ---
1. View Profile
2. Update Password
3. Delete Account
4. Logout
Select: 2
Enter new password: asdf1234@
Password updated successfully!

--- epsi's Dashboard ---
1. View Profile
2. Update Password
3. Delete Account
4. Logout
Select: 3
Are you sure? (yes/no): no

--- epsi's Dashboard ---
1. View Profile
2. Update Password
3. Delete Account
4. Logout
Select: 4
Logging out...
Action processed.

--- Smart User Management ---
1. Register
2. Login
3. Exit
Select: 3
Goodbye!
Action processed.
